# Equations of state in CmmDFT

In this example some of the basic functionalities and capabilities of the Equation of state (EOS) objects are shown. Providing a blueprint for handling the EOS objects.

EOS objects are primarily designed for the computation of equilibrium densities, pressures and chemical potentials of pure fluids and fluid mixtures. They are designed with the goal of aiding cDFT simulations, serving as platform for the calculation of bulk conditions as input for adsorption and diffusion calculations.

The calculations of the EOS are derived from the expression of the Helmholtz free energy per particle as a function of the density $\left(\frac{A(\rho)}{N}=a(\rho)\right)$ and its derivatives toward the particle density. From this quantity the pressure and chemical potential follow:

$$P(\rho, T) = k_BT\rho + \rho^{2}\frac{\partial a(\rho)}{\partial \rho}$$
$$\mu(\rho, T) = a(\rho) + \rho\frac{\partial a(\rho)}{\partial \rho}$$

## Pure fluids

As an example the modified Benedict-Webb-Rubin EOS is constructed for methane.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from cmmdft.units_constants import angstrom, boltzmann, kelvin, bar, amu, kjmol
from cmmdft.eos import ModifiedBenedictWebbRubinEOS as MBWR

In [ ]:
mass_CH4 = 16 * amu
sigma_CH4 = 3.73 * angstrom
epsilon_CH4 = 150 * boltzmann

EOS = MBWR(mass_CH4, sigma_CH4, epsilon_CH4)

# set temperature of EOS
EOS.set_temperature(300 * kelvin)

# get density at 1 bar
rho_CH4 = EOS.compute_density(pressure=np.geomspace(0.01,100, 100)*bar)
mu_CH4 = EOS.compute_chempot(rho=rho_CH4)

## Fluid mixtures

Mixture EOS objects are built on the `EOS_MIX` base class. The most common workflow is a homogeneous mixture with a fixed composition: set `homogeneous=True` and provide a mole-fraction vector `x` (or `homogeneous_fraction`). In that case, the input density is interpreted as the total mixture density, and CmmDFT distributes it among the species according to the fixed composition. `homogeneous=False` is not meant to be used directly, it is used by the relevant LDA and WDA functionals of cDFT.

This is the typical setup for bulk mixture calculations in adsorption or diffusion workflows. 


### Example: binary MBWR mixture at fixed composition

A two-component mixture can be represented by a single total density and a fixed mole-fraction vector. The EOS then returns component-wise chemical potentials while the total pressure is computed from the total density.


In [ ]:
from cmmdft.eos import ModifiedBenedictWebbRubinMixEOS as MBWRMix

# Example: CH4 / C2H6-like binary mixture at 300 K
mass_mix = np.array([16, 30]) * amu
sigma_mix = np.array([3.73, 4.10]) * angstrom
epsilon_mix = np.array([150, 180]) * boltzmann
x_mix = np.array([0.7, 0.3])

EOS_mix = MBWRMix(mass_mix, sigma_mix, epsilon_mix, homogeneous=True, x=x_mix)
EOS_mix.set_temperature(300 * kelvin)

# Pressure range used in bulk EOS calculations
pressures = np.geomspace(0.01, 100, 100) * bar
rho_total = EOS_mix.compute_density(pressure=pressures)

# Component densities for a fixed mixture composition
rho_components = EOS_mix.homogeneous_fraction[None, :] * rho_total[:, None]

# Chemical potentials of each component at the mixture density
mu_components = EOS_mix.compute_chempot(rho=rho_total)

# Total pressure for a homogeneous mixture from the total density
P_total = EOS_mix.compute_pressure(rho=rho_total)

# Show values around 1 bar for intuition
idx = np.argmin(np.abs(pressures - 1 * bar))
print('Mole fractions:', EOS_mix.homogeneous_fraction)
print('Total density at 1 bar [1/A^3]:', rho_total[idx] / (1 / angstrom**3))
print('Component densities at 1 bar [1/A^3]:', rho_components[idx] / (1 / angstrom**3))
print('Chemical potentials at 1 bar [kjmol]:', mu_components[idx] / kjmol)
print('Pressure at 1 bar [bar]:', P_total[idx] / bar)

In [ ]:
# Plot bulk-density curves for the mixture
plt.figure(figsize=(7, 4.5))
plt.plot(pressures / bar, rho_total, label='total density', linewidth=2)
plt.plot(pressures / bar, rho_components[:, 0], '--', label='component 1', linewidth=2)
plt.plot(pressures / bar, rho_components[:, 1], '-.', label='component 2', linewidth=2)
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Pressure [bar]')
plt.ylabel(r'Number density $\rho$ [1/Å$^3$]')
plt.title('MBWR mixture density vs pressure at 300 K')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
